In [3]:
import pandas as pd
import re
import numpy as np

In [4]:
df1 = pd.read_csv("../data/raw/purplle_campaign_data_with_nulls.csv")
df2 = pd.read_csv("../data/raw/nykaa_campaign_data_with_nulls.csv")
df3 = pd.read_csv("../data/raw/tira_campaign_data_with_nulls.csv")

In [5]:
df2['Brand'] = 'Nykaa'
df1['Brand'] = 'Purplle'
df3['Brand'] = 'Tira'

In [6]:
df2.drop(columns=['Campaign_ID'], inplace=True)
df1.drop(columns=['Campaign_ID'], inplace=True)
df3.drop(columns=['Campaign_ID'], inplace=True)

In [7]:
df = pd.concat([df2, df1, df3], ignore_index=True)
df.shape

(166665, 16)

In [8]:
df['Date'] = pd.to_datetime(df['Date'])

/tmp/ipykernel_27387/2394721818.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])


In [9]:
cols = ['Impressions', 'Clicks', 'Leads', 'Conversions']
df[cols] = df[cols].astype('Int64')

In [10]:
str_cols = ['Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment']

for col in str_cols:
    irregular = df[col].dropna().unique()
    irregular = [v for v in irregular if re.search(r'[^a-zA-Z\s,]', v)]
    print(col, '→', irregular)

Campaign_Type → []
Target_Audience → ['Tier 2 City Customers']
Channel_Used → []
Language → []
Customer_Segment → ['Tier 2 City Customers']


In [11]:
str_cols = ['Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment', 'Brand']

for col in str_cols:
    df[col] = (
        df[col]
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.title()
    )

In [12]:
str_cols = ['Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment']
num_cols = ['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue', 'Acquisition_Cost', 'ROI', 'Engagement_Score']

new_order = ['Brand', 'Date'] + str_cols + num_cols

df = df[new_order]

i am going to keep the null values of date because we do not have any solution to impute the missing value, but also we can get some insights usind date.that is why.

In [13]:
probs = df['Campaign_Type'].value_counts(normalize=True)
df['Campaign_Type'] = df['Campaign_Type'].apply(
    lambda x: np.random.choice(probs.index, p=np.asarray(probs, dtype=float)) if pd.isna(x) else x
)

In [14]:
probs = df['Target_Audience'].value_counts(normalize=True)
df['Target_Audience'] = df['Target_Audience'].apply(
    lambda x: np.random.choice(probs.index, p=np.asarray(probs, dtype=float)) if pd.isna(x) else x
)

In [15]:
probs = df['Language'].value_counts(normalize=True)
df['Language'] = df['Language'].apply(
    lambda x: np.random.choice(probs.index.to_numpy(), p=np.asarray(probs, dtype=float)) if pd.isna(x) else x
)

In [16]:
probs = df['Channel_Used'].value_counts(normalize=True)
df['Channel_Used'] = df['Channel_Used'].apply(
    lambda x: np.random.choice(probs.index.to_numpy(), p=np.asarray(probs, dtype=float)) if pd.isna(x) else x
)

In [17]:
probs = df['Customer_Segment'].value_counts(normalize=True)
df['Customer_Segment'] = df['Customer_Segment'].apply(
    lambda x: np.random.choice(probs.index.to_numpy(), p=np.asarray(probs, dtype=float)) if pd.isna(x) else x
)

In [18]:
df.groupby(['Brand', 'Campaign_Type'])['Duration'].median()

Brand    Campaign_Type
Nykaa    Email            17.0
         Influencer       17.0
         Paid Ads         17.0
         Seo              17.0
         Social Media     17.0
Purplle  Email            18.0
         Influencer       17.0
         Paid Ads         17.0
         Seo              18.0
         Social Media     17.0
Tira     Email            18.0
         Influencer       18.0
         Paid Ads         17.0
         Seo              17.0
         Social Media     18.0
Name: Duration, dtype: float64

In [19]:
df['Duration'] = df['Duration'].fillna(df['Duration'].median())
df['Duration'] = df['Duration'].astype(int)

In [20]:
df.groupby(['Brand', 'Campaign_Type'])['Impressions'].median()

Brand    Campaign_Type
Nykaa    Email            54954.0
         Influencer       54913.5
         Paid Ads         55559.0
         Seo              55132.0
         Social Media     55546.5
Purplle  Email            55381.0
         Influencer       54718.0
         Paid Ads         55511.5
         Seo              55041.0
         Social Media     55041.0
Tira     Email            54986.0
         Influencer       54885.5
         Paid Ads         55211.0
         Seo              54930.0
         Social Media     54937.0
Name: Impressions, dtype: Float64

In [21]:
df['Impressions'] = df['Impressions'].fillna(df['Impressions'].median())
df['Impressions'] = df['Impressions'].astype(int)

In [22]:
ctr = round((df['Clicks'] / df['Impressions']).median(), 4)
print(ctr)

0.0845


In [23]:
df['Clicks'] = df['Clicks'].fillna(round(df['Impressions'] * ctr))
df['Clicks'] = df['Clicks'].astype(int)

In [24]:
lead_rate = round((df['Leads'] / df['Clicks']).median(), 4)
print(lead_rate)

0.3977


In [25]:
df['Leads'] = df['Leads'].fillna(round(df['Clicks'] * lead_rate))
df['Leads'] = df['Leads'].astype(int)

In [26]:
(df['Leads'] > df['Clicks']).sum()   # should be 0

np.int64(37)

In [27]:
bad_rows = df[df['Leads'] > df['Clicks']]
print(len(bad_rows))
bad_rows[['Impressions', 'Clicks', 'Leads']]

37


,Impressions,Clicks,Leads
11394,55113,4657,5796
13544,55113,4657,6857
14118,28862,2439,2450
16472,55113,4657,5068
19446,69186,5846,6063
23596,55113,4657,5985
32081,45148,3815,3935
43359,55113,4657,4754
44020,65285,5517,5558
49906,55113,4657,5431


In [28]:
df = df[df['Leads'] <= df['Clicks']]

In [29]:
df = df.reset_index(drop=True)

In [30]:
conv_rate = round((df['Conversions'] / df['Leads']).median(), 4)
print(conv_rate)

0.5488


In [31]:
df['Conversions'] = df['Conversions'].fillna(round(df['Leads'] * conv_rate))
df['Conversions'] = df['Conversions'].astype(int)

In [32]:
(df['Conversions'] > df['Leads']).sum()

np.int64(341)

In [33]:
bad_rows = df[df['Conversions'] > df['Leads']]
print(len(bad_rows))
bad_rows[['Clicks', 'Leads', 'Conversions']].head(10)

341


,Clicks,Leads,Conversions
901,4732,1882,1965
1597,5004,1990,2016
2003,6556,2607,4502
2776,3068,1220,1367
3121,6810,2708,2744
3232,5304,2109,2432
3562,4651,1850,1903
3887,4502,1790,1906
4000,2252,896,906
4374,5902,2347,2521


In [34]:
df = df[df['Conversions'] <= df['Leads']]
df = df.reset_index(drop=True)

In [35]:
df[['Impressions','Clicks','Leads','Conversions','Duration','Acquisition_Cost','Revenue']].corr()['Acquisition_Cost']


Impressions        -0.424094
Clicks             -0.476795
Leads              -0.471674
Conversions        -0.462922
Duration           -0.000512
Acquisition_Cost    1.000000
Revenue            -0.405833
Name: Acquisition_Cost, dtype: float64

In [36]:
df[['Impressions','Clicks','Leads','Conversions','Duration','Acquisition_Cost','Revenue']].corr()['Revenue']

Impressions         0.486846
Clicks              0.707439
Leads               0.803052
Conversions         0.874945
Duration           -0.000447
Acquisition_Cost   -0.405833
Revenue             1.000000
Name: Revenue, dtype: float64

In [37]:
df.groupby(['Brand', 'Campaign_Type'])['Acquisition_Cost'].median()

Brand    Campaign_Type
Nykaa    Email            207.630
         Influencer       206.890
         Paid Ads         207.710
         Seo              208.740
         Social Media     207.410
Purplle  Email            209.555
         Influencer       211.850
         Paid Ads         208.295
         Seo              213.185
         Social Media     210.380
Tira     Email            210.140
         Influencer       208.865
         Paid Ads         204.200
         Seo              211.555
         Social Media     209.570
Name: Acquisition_Cost, dtype: float64

In [38]:
df['Acquisition_Cost'] = df['Acquisition_Cost'].fillna(df['Acquisition_Cost'].median())
df['Acquisition_Cost'] = df['Acquisition_Cost'].astype(int)

In [39]:
float_cols = ['Revenue', 'Acquisition_Cost', 'ROI', 'Engagement_Score', 'Duration']
df[float_cols] = df[float_cols].round(2)

In [40]:
revenue_per_conversion = (df['Revenue'] / df['Conversions']).median()
df['Revenue'] = df['Revenue'].fillna(round(df['Conversions'] * revenue_per_conversion, 2))
df['Revenue'] = df['Revenue'].astype(int)

In [41]:
df['Revenue'].describe() 

count    1.662870e+05
mean     5.125645e+05
std      4.831510e+05
min      3.895000e+03
25%      1.784760e+05
50%      3.598230e+05
75%      6.847470e+05
max      4.579910e+06
Name: Revenue, dtype: float64

In [42]:
df['Acquisition_Cost'].describe() 

count    166287.000000
mean        368.625413
std         525.053247
min           9.000000
25%         111.000000
50%         209.000000
75%         411.000000
max       15473.000000
Name: Acquisition_Cost, dtype: float64

In [43]:
df['Total_Cost'] = df['Acquisition_Cost'] * df['Conversions']

df['ROI'] = (df['Revenue'] - df['Total_Cost']) / df['Total_Cost']
df['ROI'] = df['ROI'].round(2)

In [44]:
engagement_per_lead = (df['Engagement_Score'] / df['Leads']).median()
df['Engagement_Score'] = df['Engagement_Score'].fillna(round(df['Leads'] * engagement_per_lead, 2))

In [45]:
df['Engagement_Score'].describe() 

count    166287.000000
mean         13.795027
std           6.631456
min           0.470000
25%           8.260000
50%          13.500000
75%          18.810000
max          67.820000
Name: Engagement_Score, dtype: float64

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 166287 entries, 0 to 166286
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Brand             166287 non-null  str           
 1   Date              158209 non-null  datetime64[us]
 2   Campaign_Type     166287 non-null  str           
 3   Target_Audience   166287 non-null  str           
 4   Channel_Used      166287 non-null  str           
 5   Language          166287 non-null  str           
 6   Customer_Segment  166287 non-null  str           
 7   Duration          166287 non-null  int64         
 8   Impressions       166287 non-null  int64         
 9   Clicks            166287 non-null  int64         
 10  Leads             166287 non-null  int64         
 11  Conversions       166287 non-null  int64         
 12  Revenue           166287 non-null  int64         
 13  Acquisition_Cost  166287 non-null  int64         
 14  ROI            

* Brand column added, Campaign_ID dropped
* Date → datetime, normalized
* Funnel columns → Int64
* String columns → stripped, spaced, title-cased
* Columns reordered (Brand, Date, strings, numerics)
* Categorical nulls → proportional fill (Campaign_Type, Target_Audience, Channel_Used, Language, Customer_Segment)
* Funnel nulls → ratio-based fill (Impressions→Clicks→Leads→Conversions), funnel-logic violations dropped
* Revenue → ratio fill from Conversions
* Acquisition_Cost → group/median fill
* ROI → recomputed correctly using Total_Cost
* Engagement_Score → ratio fill from Leads
* Floats rounded to 2 decimals

In [47]:
print(df.isnull().sum())

assert (df['Clicks'] <= df['Impressions']).all()
assert (df['Leads'] <= df['Clicks']).all()
assert (df['Conversions'] <= df['Leads']).all()


print(df.duplicated().sum())

print(df.shape)
df.head()

Brand                  0
Date                8078
Campaign_Type          0
Target_Audience        0
Channel_Used           0
Language               0
Customer_Segment       0
Duration               0
Impressions            0
Clicks                 0
Leads                  0
Conversions            0
Revenue                0
Acquisition_Cost       0
ROI                    0
Engagement_Score       0
Total_Cost             0
dtype: int64
0
(166287, 17)


,Brand,Date,Campaign_Type,Target_Audience,Channel_Used,Language,Customer_Segment,Duration,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Engagement_Score,Total_Cost
0,Nykaa,2025-04-29,Social Media,College Students,"Whatsapp, Youtube",Hindi,College Students,21,57804,6156,3616,2355,1867515,209,2.79,20.98,492195
1,Nykaa,2025-04-06,Paid Ads,Tier 2 City Customers,Youtube,Hindi,College Students,18,91801,3321,1971,1357,1046247,180,3.28,7.24,244260
2,Nykaa,2025-01-14,Influencer,Youth,"Whatsapp, Google, Youtube",English,College Students,23,15536,2182,952,755,197055,90,1.90,25.03,67950
3,Nykaa,2025-06-04,Email,Working Women,"Youtube, Facebook, Instagram",Hindi,College Students,18,88114,8413,2231,947,376906,249,0.60,13.15,235803
4,Nykaa,2024-12-29,Paid Ads,College Students,"Facebook, Instagram",Hindi,Tier 2 City Customers,10,96871,3743,2060,1258,518296,228,0.81,7.29,286824


In [48]:
df.to_csv('../data/cleaned/cleaned_data.csv', index=False)

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 166287 entries, 0 to 166286
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Brand             166287 non-null  str           
 1   Date              158209 non-null  datetime64[us]
 2   Campaign_Type     166287 non-null  str           
 3   Target_Audience   166287 non-null  str           
 4   Channel_Used      166287 non-null  str           
 5   Language          166287 non-null  str           
 6   Customer_Segment  166287 non-null  str           
 7   Duration          166287 non-null  int64         
 8   Impressions       166287 non-null  int64         
 9   Clicks            166287 non-null  int64         
 10  Leads             166287 non-null  int64         
 11  Conversions       166287 non-null  int64         
 12  Revenue           166287 non-null  int64         
 13  Acquisition_Cost  166287 non-null  int64         
 14  ROI            